In [2]:
import os
import pandas as pd
import requests
from datetime import datetime

api_key = os.getenv("API_KEY")         
cities = ["Louisville", "Trivandrum", "Reykjavik", "Dubai"]

DAYS_OF_FORECAST = 3  

def get_weather_data(cities, api_key):
    baseurl = "http://api.weatherapi.com/v1/"
    today = pd.Timestamp.now().strftime("%Y-%m-%d")
    data_list = []
    
    for city in cities:
        try:
            # Current weather
            curr_url = f"{baseurl}current.json?key={api_key}&q={city}"
            curr_data = requests.get(curr_url).json()['current']
            
            current_summary = (f"{curr_data['condition']['text']} - "
                              f"Temp: {curr_data['temp_f']}°F, "
                              f"Humidity: {curr_data['humidity']}%, "
                              f"Wind: {curr_data['wind_mph']} MPH")

            # 3-day Forecast
            forecast_url = f"{baseurl}forecast.json?key={api_key}&q={city}&days={DAYS_OF_FORECAST}"
            forecast_days = requests.get(forecast_url).json()['forecast']['forecastday']
            
            forecast_summary = []
            for day in forecast_days:
                date_obj = datetime.strptime(day['date'], '%Y-%m-%d')
                formatted_date = date_obj.strftime('%A, %b %d')
                forecast_summary.append(
                    f"{formatted_date}: High {day['day']['maxtemp_f']}°F / "
                    f"Low {day['day']['mintemp_f']}°F - {day['day']['condition']['text']}"
                )
            forecast_str = " | ".join(forecast_summary)

            # Astronomy
            astro_url = f"{baseurl}astronomy.json?key={api_key}&q={city}&dt={today}"
            astro_data = requests.get(astro_url).json()['astronomy']['astro']
            
            astro_summary = (f"Sunrise: {astro_data['sunrise']} | "
                           f"Sunset: {astro_data['sunset']} | "
                           f"Moon: {astro_data['moon_phase']}")

            data_list.append({
                'City': city,
                'Current_Conditions': current_summary,
                'Forecast_3_Days': forecast_str,
                'Astronomy': astro_summary,
                'Timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            
        except Exception as e:
            print(f"Error fetching data for {city}: {e}")
            data_list.append({
                'City': city,
                'Current_Conditions': f"Error: {str(e)}",
                'Forecast_3Days': "",
                'Astronomy': "",
                'Timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
    
    return pd.DataFrame(data_list)



if __name__ == "__main__":
    df = get_weather_data(cities, api_key)
    
    today_str = pd.Timestamp.now().strftime("%Y-%m-%d")
    filename = f"{today_str}_cities_weather.csv"
    
    df.to_csv(filename, index=False)